# 酶活性数据存储脚本

**用途**：把当前内存里的实验数据表保存成一个 CSV 文件，方便长期留存、用 Excel 查看、或以后再用 Python 读回来分析。

**使用步骤**：从上到下依次运行每个代码单元格（菜单 `Run` → `Run All Cells`，或按 `Shift+Enter` 逐个运行）。

**存储策略**：
- 格式：CSV（纯文本，Excel 双击可开，任何软件都能读）
- 位置：`data/` 子文件夹
- 文件名：固定的 `activity.csv`，**每次运行都会覆盖上一份**，只保留最新数据

**结构**：配置区（第 2 格）→ 准备数据（第 3 格）→ 体检（第 4 格）→ 保存（第 5 格）→ 回读验证（第 6 格）。

> 注意：本脚本**只负责保存，不做任何统计分析**；分析仍然放在 `first_analyze.ipynb` 里做。

In [1]:
# ========== 1. 导入需要的库 ==========
import pandas as pd               # 表格数据处理
import numpy as np                # 用 np.nan 表示“未获得测量结果”
from pathlib import Path          # 路径操作，比用字符串拼路径更可靠（Windows/Mac 通用）
from datetime import datetime     # 记录保存时间

In [2]:
# ========== 2. 配置区：以后要改的通常只有这几行 ==========
DATA_DIR  = Path("data")        # CSV 存放的文件夹（相对本 notebook 所在目录）
FILE_NAME = "activity.csv"      # 固定文件名：每次运行都会覆盖上一份
NA_REP    = ""                  # 缺失值在 CSV 里写成空字段（在 Excel 中就是一个空格子）

print("数据目录：", DATA_DIR.resolve())
print("目标文件：", (DATA_DIR / FILE_NAME).resolve())

数据目录： C:\A_work\Project_1\data
目标文件： C:\A_work\Project_1\data\activity.csv


In [3]:
# ========== 3. 准备要保存的数据表 ==========
# 这里沿用 first_analyze.ipynb 里的实验数据。
# 实际使用时，把这一段换成你自己的数据来源（例如用 pd.read_excel() 读入仪器导出的表）。
data = pd.DataFrame({
    "sample_id": [1, 2, 3, 4, 5, 6],
    "group": ["control"] * 3 + ["treatment"] * 3,
    "activity": [90, 100, 110, 120, np.nan, 140],   # 5 号样品未测得结果
})

print("数据形状（行, 列）：", data.shape)
data.head(10)

数据形状（行, 列）： (6, 3)


,sample_id,group,activity
0,1,control,90.0
1,2,control,100.0
2,3,control,110.0
3,4,treatment,120.0
4,5,treatment,NaN
5,6,treatment,140.0


In [4]:
# ========== 4. 保存前体检：先看清数据，再决定要不要存 ==========
# 存储脚本最容易犯的错不是“写不出来”，而是把有问题的数据存进去还不知道。
print("各列数据类型：")
print(data.dtypes)

print("\n各列缺失值个数：")
print(data.isna().sum())

# 按分组看每个组到底测到了几个样品：
#   count = 有效数据个数（跳过 NaN）；size = 总行数（不跳 NaN）
check = data.groupby("group")["activity"].agg(["count", "size"])
check["缺失值个数"] = check["size"] - check["count"]
print("\n各组数据完整性：")
print(check)

print("\n提示：count 与 size 不一致的组，说明该组有样品没测到结果。")

各列数据类型：
sample_id      int64
group            str
activity     float64
dtype: object

各列缺失值个数：
sample_id    0
group        0
activity     1
dtype: int64

各组数据完整性：
           count  size  缺失值个数
group                        
control        3     3      0
treatment      2     3      1

提示：count 与 size 不一致的组，说明该组有样品没测到结果。


In [ ]:
# ========== 5. 覆盖保护 + 保存 ==========
DATA_DIR.mkdir(parents=True, exist_ok=True)   # 文件夹不存在就自动创建（已存在不会报错）

target = DATA_DIR / FILE_NAME

# 本脚本采用“固定文件名、每次覆盖”策略，所以先检查是不是真的会覆盖掉已有数据，
# 让这件事不要“悄悄发生”，而是明确打印出来给你看。
if target.exists():
    old = pd.read_csv(target)
    print(f"注意：{target} 已存在（{len(old)} 行），本次保存将覆盖它。")
else:
    print(f"目标文件不存在，将新建：{target}")

# index=False   ：不把 DataFrame 的行号（0,1,2...）写进 CSV
# encoding="utf-8-sig"：让 Excel 打开时中文不乱码
# na_rep=NA_REP ：缺失值写成空字段，而不是 "NaN" 这几个字母
data.to_csv(target, index=False, encoding="utf-8-sig", na_rep=NA_REP)

print(f"\n保存完成 | 时间：{datetime.now():%Y-%m-%d %H:%M:%S}")
print("完整路径：", target.resolve())

目标文件不存在，将新建：data\activity.csv

保存完成 | 时间：2026-09-27 22:09:53
完整路径： C:\A_work\Project_1\data\activity.csv


In [ ]:
# ========== 6. 回读验证：把刚保存的文件读回来，确认没写坏 ==========
saved = pd.read_csv(DATA_DIR / FILE_NAME)

print("原表形状：", data.shape, " 回读形状：", saved.shape)
print("行数一致：", saved.shape[0] == data.shape[0])
print("列名一致：", list(saved.columns) == list(data.columns))

print("\n缺失值个数对比（保存前 / 读回后）：")
print(pd.DataFrame({
    "保存前": data.isna().sum(),
    "读回后": saved.isna().sum(),
}))

saved.head(10)

原表形状： (6, 3)  回读形状： (6, 3)
行数一致： True
列名一致： True

缺失值个数对比（保存前 / 读回后）：
           保存前  读回后
sample_id    0    0
group        0    0
activity     1    1


,sample_id,group,activity
0,1,control,90.0
1,2,control,100.0
2,3,control,110.0
3,4,treatment,120.0
4,5,treatment,NaN
5,6,treatment,140.0


## 关于缺失值的说明（结合上次的结论）

1. **CSV 里的“空缺”是存活下来的。**
   缺失值写出时是空字段（`5,treatment,`），读回来 `pandas` 仍然识别为 `NaN`。
   这一点由第 6 格的回读验证保证：保存前 1 个缺失，读回后还是 1 个缺失。

2. **不要用 `fillna(0)` 来“解决”空缺。**
   空缺表示“没测到结果”，不等于“酶活性为 0”。如果先 `fillna(0)` 再保存，
   处理组的均值会从 130.0 被拉到 86.7，而且 `count` 会从 2 变成 3，
   缺失的痕迹被彻底抹掉 —— 以后任何人都看不出这批数据少了一个样品。
   **原始数据文件里务必保留空缺本身。**

3. **文件里的行数 ≠ 有效样本量。**
   用文件算任何统计量前，先像第 4 格那样看一遍 `count` 与 `size`。

## 常见改动

| 想做的改动 | 怎么改 |
| --- | --- |
| 换存储位置 | 改第 2 格的 `DATA_DIR`，例如 `Path("D:/lab_data")` |
| 改成保留历史版本 | 把第 2 格的 `FILE_NAME` 改成带时间戳的名字，例如在第 5 格保存前加一行：`FILE_NAME = f"activity_{datetime.now():%Y%m%d_%H%M%S}.csv"` |
| 数据来自 Excel | 把第 3 格换成 `data = pd.read_excel("原始数据.xlsx")` |
| 同时保存分组汇总 | 在第 4 格后加 `check.to_csv(DATA_DIR / "summary.csv", encoding="utf-8-sig")` |

> 采用“固定文件名、每次覆盖”意味着上一版数据无法找回。
> 如果这批数据比较重要，建议把 `data/activity.csv` 定期另存一份到别处，或改用带时间戳的文件名。